In [3]:

!pip install pandas numpy scikit-learn
!pip install fastapi uvicorn joblib pyngrok


In [4]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\Pujitha\G-1721\data\raw\Agricultural_Yield_Final-1.csv")
print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)
df.head()

Dataset loaded successfully!
Dataset Shape: (926, 50)


,country,year,crop,yield,yield_unit,flag,flag_description,crop_area_harvested,crop_area_unit,fertilizer__nutrient_nitrogen_n_total__agricultural_use,...,land__land_area__area,land__permanent_crops__area,land__permanent_meadows_and_pastures__area,land__temporary_crops__area,land__temporary_fallow__area,land__temporary_meadows_and_pastures__area,pesticide__pesticides_total__agricultural_use,pesticide__pesticides_total__use_per_area_of_cropland,pesticide__pesticides_total__use_per_capita,pesticide__pesticides_total__use_per_value_of_agricultural_production
0,India,2014,"Almonds, in shell",619.0,kg/ha,A,Official value,21000.0,ha,16949600.0,...,297319.0,13040.02,10265.0,119387.92,24851.0,12184.06,40094.0,0.24,0.03,0.10
1,India,2015,"Almonds, in shell",476.2,kg/ha,A,Official value,21000.0,ha,17372300.0,...,297319.0,12273.88,10262.0,119636.62,26181.0,11321.50,40094.0,0.24,0.03,0.10
2,India,2016,"Almonds, in shell",678.9,kg/ha,A,Official value,11685.0,ha,16735400.0,...,297319.0,12464.95,10261.0,119662.07,26718.0,10471.98,40094.0,0.24,0.03,0.10
3,India,2017,"Almonds, in shell",584.8,kg/ha,A,Official value,12075.0,ha,16958000.0,...,297319.0,12190.51,10340.0,119911.45,26356.0,10437.03,40094.0,0.24,0.03,0.10
4,India,2018,"Almonds, in shell",1293.3,kg/ha,A,Official value,10584.0,ha,17628200.0,...,297319.0,12705.96,10338.0,119343.25,26430.0,10298.79,40094.0,0.24,0.03,0.09


In [5]:
print("Dataset Information:")
df.info()
print("\nMissing Values:")
print(df.isnull().sum())
print("\nColumn Names:")
print(df.columns.tolist())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 926 entries, 0 to 925
Data columns (total 50 columns):
 #   Column                                                                               Non-Null Count  Dtype  
---  ------                                                                               --------------  -----  
 0   country                                                                              926 non-null    object 
 1   year                                                                                 926 non-null    int64  
 2   crop                                                                                 926 non-null    object 
 3   yield                                                                                926 non-null    float64
 4   yield_unit                                                                           926 non-null    object 
 5   flag                                                                   

In [6]:
print("Target column: yield")
print(df["yield"].describe())
print("\nYield unit distribution:")
print(df["yield_unit"].value_counts())

Target column: yield
count      926.000000
mean      9117.385853
std      11783.328321
min        205.100000
25%       1272.000000
50%       4002.500000
75%      12465.100000
max      84906.300000
Name: yield, dtype: float64

Yield unit distribution:
yield_unit
kg/ha    915
No/An     11
Name: count, dtype: int64


In [7]:
df = df[df["yield_unit"] == "kg/ha"].copy()
df = df.dropna(subset=["yield"])
print("Dataset after cleaning:", df.shape)
print("Missing target values:", df["yield"].isnull().sum())

Dataset after cleaning: (915, 50)
Missing target values: 0


In [8]:
y = df["yield"]
drop_columns = [
    "yield",
    "yield_unit",
    "country",
    "flag",
    "flag_description",
    "crop_area_unit"
]
X = df.drop(columns=drop_columns)
print("Input features:", X.shape)
print("Target:", y.shape)

Input features: (915, 44)
Target: (915,)


In [9]:
high_missing = X.columns[X.isnull().mean() > 0.40]
X = X.drop(columns=high_missing)
constant_columns = [
    col for col in X.columns
    if X[col].nunique(dropna=False) <= 1
]
X = X.drop(columns=constant_columns)
print("Remaining features:", X.shape[1])
print("Features:", X.columns.tolist())

Remaining features: 36
Features: ['year', 'crop', 'crop_area_harvested', 'fertilizer__nutrient_nitrogen_n_total__agricultural_use', 'fertilizer__nutrient_nitrogen_n_total__export_quantity', 'fertilizer__nutrient_nitrogen_n_total__import_quantity', 'fertilizer__nutrient_nitrogen_n_total__production', 'fertilizer__nutrient_nitrogen_n_total__use_per_area_of_cropland', 'fertilizer__nutrient_nitrogen_n_total__use_per_capita', 'fertilizer__nutrient_nitrogen_n_total__use_per_value_of_agricultural_production', 'fertilizer__nutrient_phosphate_p2o5_total__agricultural_use', 'fertilizer__nutrient_phosphate_p2o5_total__export_quantity', 'fertilizer__nutrient_phosphate_p2o5_total__import_quantity', 'fertilizer__nutrient_phosphate_p2o5_total__production', 'fertilizer__nutrient_phosphate_p2o5_total__use_per_area_of_cropland', 'fertilizer__nutrient_phosphate_p2o5_total__use_per_capita', 'fertilizer__nutrient_phosphate_p2o5_total__use_per_value_of_agricultural_production', 'fertilizer__nutrient_potash_

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()
print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Numerical features: ['year', 'crop_area_harvested', 'fertilizer__nutrient_nitrogen_n_total__agricultural_use', 'fertilizer__nutrient_nitrogen_n_total__export_quantity', 'fertilizer__nutrient_nitrogen_n_total__import_quantity', 'fertilizer__nutrient_nitrogen_n_total__production', 'fertilizer__nutrient_nitrogen_n_total__use_per_area_of_cropland', 'fertilizer__nutrient_nitrogen_n_total__use_per_capita', 'fertilizer__nutrient_nitrogen_n_total__use_per_value_of_agricultural_production', 'fertilizer__nutrient_phosphate_p2o5_total__agricultural_use', 'fertilizer__nutrient_phosphate_p2o5_total__export_quantity', 'fertilizer__nutrient_phosphate_p2o5_total__import_quantity', 'fertilizer__nutrient_phosphate_p2o5_total__production', 'fertilizer__nutrient_phosphate_p2o5_total__use_per_area_of_cropland', 'fertilizer__nutrient_phosphate_p2o5_total__use_per_capita', 'fertilizer__nutrient_phosphate_p2o5_total__use_per_value_of_agricultural_production', 'fertilizer__nutrient_potash_k2o_total__agricultur

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (732, 36)
Testing data: (183, 36)


In [12]:
import joblib
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    ))
])
model.fit(X_train, y_train)
print("Model trained successfully!")

Model trained successfully!


In [13]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 785.3274022219085
RMSE: 2004.1639361652317
R2 Score: 0.9745035006075785


In [14]:
joblib.dump({
    "model": model,
    "features": X.columns.tolist()
}, "model.pkl")
print("Model saved successfully!")

Model saved successfully!


In [15]:
import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv(
    "../data/processed/cleaned_data.csv",
    index=False
)

print("Processed CSV saved successfully!")

Processed CSV saved successfully!


In [16]:
import os

print("Current location:", os.getcwd())

Current location: C:\Users\Pujitha\G-1721\notebooks


In [17]:
import os
import joblib

model_folder = r"C:\Users\Pujitha\G-1721\models"
os.makedirs(model_folder, exist_ok=True)

joblib.dump({
    "model": model,
    "features": X.columns.tolist()
}, os.path.join(model_folder, "model.pkl"))

print("Model saved successfully!")
print(os.listdir(model_folder))

Model saved successfully!
['model.pkl']


In [18]:
print("Crop data type:", X["crop"].dtype)

print("\nSample crop values:")
print(X["crop"].head(10))

print("\nUnique crop values:")
print(X["crop"].nunique())

Crop data type: object

Sample crop values:
0    Almonds, in shell
1    Almonds, in shell
2    Almonds, in shell
3    Almonds, in shell
4    Almonds, in shell
5    Almonds, in shell
6    Almonds, in shell
7    Almonds, in shell
8    Almonds, in shell
9    Almonds, in shell
Name: crop, dtype: object

Unique crop values:
84


In [19]:
print("Model type:", type(model))

Model type: <class 'sklearn.pipeline.Pipeline'>


In [20]:
print("Crop encoding information:")

if "preprocessor" in globals():
    print(preprocessor)
elif "encoder" in globals():
    print(encoder)
else:
    print("No variable named preprocessor or encoder found.")

Crop encoding information:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['year', 'crop_area_harvested',
                                  'fertilizer__nutrient_nitrogen_n_total__agricultural_use',
                                  'fertilizer__nutrient_nitrogen_n_total__export_quantity',
                                  'fertilizer__nutrient_nitrogen_n_total__import_quantity',
                                  'fertilizer__nutrient_nitrogen_n_total__produc...
                                  'land__agriculture__area',
                                  'land__agriculture_area_under_organic_agric__area',
                                  'land__arable_land__area',
                                  'land__country_area__area',
                                  'land__cropland__area',
                               

In [21]:
print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['year',
                                                   'crop_area_harvested',
                                                   'fertilizer__nutrient_nitrogen_n_total__agricultural_use',
                                                   'fertilizer__nutrient_nitrogen_n_total__export_quantity',
                                                   'fertilizer__nutrient_nitrogen_n_total__import_quantity',
                                                   'fertilizer__n...
                                                   'land__country_area__area',
                                                   'land__cropland__area',
                                            

In [22]:
print("Pipeline steps:")

for name, step in model.named_steps.items():
    print(name, ":", type(step))

Pipeline steps:
preprocessor : <class 'sklearn.compose._column_transformer.ColumnTransformer'>
regressor : <class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [23]:
from sklearn.compose import ColumnTransformer

for name, step in model.named_steps.items():
    if isinstance(step, ColumnTransformer):
        print("Found ColumnTransformer:", name)
        print(step)

Found ColumnTransformer: preprocessor
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['year', 'crop_area_harvested',
                                  'fertilizer__nutrient_nitrogen_n_total__agricultural_use',
                                  'fertilizer__nutrient_nitrogen_n_total__export_quantity',
                                  'fertilizer__nutrient_nitrogen_n_total__import_quantity',
                                  'fertilizer__nutrient_nitrogen_n_total__produc...
                                  'land__agriculture__area',
                                  'land__agriculture_area_under_organic_agric__area',
                                  'land__arable_land__area',
                                  'land__country_area__area',
                                  'land__cropland__area',
                    

In [24]:
print(X.head(1).to_dict(orient="records"))

[{'year': 2014, 'crop': 'Almonds, in shell', 'crop_area_harvested': 21000.0, 'fertilizer__nutrient_nitrogen_n_total__agricultural_use': 16949600.0, 'fertilizer__nutrient_nitrogen_n_total__export_quantity': 23776.0, 'fertilizer__nutrient_nitrogen_n_total__import_quantity': 4180999.0, 'fertilizer__nutrient_nitrogen_n_total__production': 12433700.0, 'fertilizer__nutrient_nitrogen_n_total__use_per_area_of_cropland': 100.02, 'fertilizer__nutrient_nitrogen_n_total__use_per_capita': 12.92, 'fertilizer__nutrient_nitrogen_n_total__use_per_value_of_agricultural_production': 43.79, 'fertilizer__nutrient_phosphate_p2o5_total__agricultural_use': 6098900.0, 'fertilizer__nutrient_phosphate_p2o5_total__export_quantity': 20552.0, 'fertilizer__nutrient_phosphate_p2o5_total__import_quantity': 1953601.0, 'fertilizer__nutrient_phosphate_p2o5_total__production': 4118900.0, 'fertilizer__nutrient_phosphate_p2o5_total__use_per_area_of_cropland': 35.99, 'fertilizer__nutrient_phosphate_p2o5_total__use_per_capita

In [25]:
crop_list = sorted(X["crop"].dropna().unique().tolist())

print("Total crops:", len(crop_list))
print(crop_list)

Total crops: 84
['Almonds, in shell', 'Anise, badian, coriander, cumin, caraway, fennel and juniper berries, raw', 'Apples', 'Apricots', 'Areca nuts', 'Bananas', 'Barley', 'Beans, dry', 'Cabbages', 'Cantaloupes and other melons', 'Carrots and turnips', 'Cashew nuts, in shell', 'Cassava, fresh', 'Castor oil seeds', 'Cauliflowers and broccoli', 'Cherries', 'Chick peas, dry', 'Chillies and peppers, dry (Capsicum spp., Pimenta spp.), raw', 'Chillies and peppers, green (Capsicum spp. and Pimenta spp.)', 'Cocoa beans', 'Coconuts, in shell', 'Coffee, green', 'Cucumbers and gherkins', 'Eggplants (aubergines)', 'Figs', 'Ginger, raw', 'Grapes', 'Green garlic', 'Groundnuts, excluding shelled', 'Jute, raw or retted', 'Kenaf, and other textile bast fibres, raw or retted', 'Kiwi fruit', 'Lemons and limes', 'Lentils, dry', 'Lettuce and chicory', 'Linseed', 'Maize (corn)', 'Mangoes, guavas and mangosteens', 'Millet', 'Natural rubber in primary forms', 'Nutmeg, mace, cardamoms, raw', 'Okra', 'Onions an

In [27]:
# Get sample input rows for three crops

selected_crops = ["Rice", "Wheat", "Maize (corn)"]

for crop_name in selected_crops:
    sample = X[X["crop"] == crop_name]

    if not sample.empty:
        print("\nCrop:", crop_name)
        display(sample.head(1))
    else:
        print(crop_name, "not found")


Crop: Rice


,year,crop,crop_area_harvested,fertilizer__nutrient_nitrogen_n_total__agricultural_use,fertilizer__nutrient_nitrogen_n_total__export_quantity,fertilizer__nutrient_nitrogen_n_total__import_quantity,fertilizer__nutrient_nitrogen_n_total__production,fertilizer__nutrient_nitrogen_n_total__use_per_area_of_cropland,fertilizer__nutrient_nitrogen_n_total__use_per_capita,fertilizer__nutrient_nitrogen_n_total__use_per_value_of_agricultural_production,...,land__arable_land__area,land__country_area__area,land__cropland__area,land__inland_waters__area,land__permanent_crops__area,land__permanent_meadows_and_pastures__area,land__temporary_crops__area,land__temporary_fallow__area,land__temporary_meadows_and_pastures__area,pesticide__pesticides_total__use_per_value_of_agricultural_production
729,2014,Rice,44515890.0,16949600.0,23776.0,4180999.0,12433700.0,100.02,12.92,43.79,...,156422.978,328726.0,169463.0,31407.0,13040.02,10265.0,119387.92,24851.0,12184.06,0.1



Crop: Wheat


,year,crop,crop_area_harvested,fertilizer__nutrient_nitrogen_n_total__agricultural_use,fertilizer__nutrient_nitrogen_n_total__export_quantity,fertilizer__nutrient_nitrogen_n_total__import_quantity,fertilizer__nutrient_nitrogen_n_total__production,fertilizer__nutrient_nitrogen_n_total__use_per_area_of_cropland,fertilizer__nutrient_nitrogen_n_total__use_per_capita,fertilizer__nutrient_nitrogen_n_total__use_per_value_of_agricultural_production,...,land__arable_land__area,land__country_area__area,land__cropland__area,land__inland_waters__area,land__permanent_crops__area,land__permanent_meadows_and_pastures__area,land__temporary_crops__area,land__temporary_fallow__area,land__temporary_meadows_and_pastures__area,pesticide__pesticides_total__use_per_value_of_agricultural_production
904,2014,Wheat,30470000.0,16949600.0,23776.0,4180999.0,12433700.0,100.02,12.92,43.79,...,156422.978,328726.0,169463.0,31407.0,13040.02,10265.0,119387.92,24851.0,12184.06,0.1



Crop: Maize (corn)


,year,crop,crop_area_harvested,fertilizer__nutrient_nitrogen_n_total__agricultural_use,fertilizer__nutrient_nitrogen_n_total__export_quantity,fertilizer__nutrient_nitrogen_n_total__import_quantity,fertilizer__nutrient_nitrogen_n_total__production,fertilizer__nutrient_nitrogen_n_total__use_per_area_of_cropland,fertilizer__nutrient_nitrogen_n_total__use_per_capita,fertilizer__nutrient_nitrogen_n_total__use_per_value_of_agricultural_production,...,land__arable_land__area,land__country_area__area,land__cropland__area,land__inland_waters__area,land__permanent_crops__area,land__permanent_meadows_and_pastures__area,land__temporary_crops__area,land__temporary_fallow__area,land__temporary_meadows_and_pastures__area,pesticide__pesticides_total__use_per_value_of_agricultural_production
407,2014,Maize (corn),9319980.0,16949600.0,23776.0,4180999.0,12433700.0,100.02,12.92,43.79,...,156422.978,328726.0,169463.0,31407.0,13040.02,10265.0,119387.92,24851.0,12184.06,0.1


In [28]:
import json

selected_crops = ["Rice", "Wheat", "Maize (corn)"]

for crop_name in selected_crops:
    sample = X[X["crop"] == crop_name].head(1)

    record = sample.to_dict(orient="records")[0]

    # Convert NumPy values into normal Python values
    record = {
        key: value.item() if hasattr(value, "item") else value
        for key, value in record.items()
    }

    filename = crop_name.replace(" ", "_").replace("(", "").replace(")", "") + ".json"

    with open(filename, "w") as file:
        json.dump(record, file, indent=4)

    print(filename, "created successfully!")

Rice.json created successfully!
Wheat.json created successfully!
Maize_corn.json created successfully!


In [29]:
import os
print(os.getcwd())

C:\Users\Pujitha\G-1721\notebooks


In [30]:
with open("Rice.json", "r") as file:
    print(file.read())

{
    "year": 2014,
    "crop": "Rice",
    "crop_area_harvested": 44515890.0,
    "fertilizer__nutrient_nitrogen_n_total__agricultural_use": 16949600.0,
    "fertilizer__nutrient_nitrogen_n_total__export_quantity": 23776.0,
    "fertilizer__nutrient_nitrogen_n_total__import_quantity": 4180999.0,
    "fertilizer__nutrient_nitrogen_n_total__production": 12433700.0,
    "fertilizer__nutrient_nitrogen_n_total__use_per_area_of_cropland": 100.02,
    "fertilizer__nutrient_nitrogen_n_total__use_per_capita": 12.92,
    "fertilizer__nutrient_nitrogen_n_total__use_per_value_of_agricultural_production": 43.79,
    "fertilizer__nutrient_phosphate_p2o5_total__agricultural_use": 6098900.0,
    "fertilizer__nutrient_phosphate_p2o5_total__export_quantity": 20552.0,
    "fertilizer__nutrient_phosphate_p2o5_total__import_quantity": 1953601.0,
    "fertilizer__nutrient_phosphate_p2o5_total__production": 4118900.0,
    "fertilizer__nutrient_phosphate_p2o5_total__use_per_area_of_cropland": 35.99,
    "fer

In [31]:
import requests
import json

with open("Wheat.json", "r") as file:
    wheat_data = json.load(file)

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=wheat_data
)

print("Status Code:", response.status_code)
print("Prediction:", response.json())

Status Code: 200
Prediction: {'crop': 'Wheat', 'year': 2014, 'predicted_yield': 3301.539105555557}
